# Moshi Compression — Phase 3: Attention Polish

**Goal.** Add per-layer attention-map KL between student and teacher while
preserving the Phase-1 hidden alignment and Phase-2 text-logit alignment.

**Loss.**
- Phase-1 lock: hidden cosine, weight 1.0
- Phase-2 lock: sparse top-256 text JSD, weight 0.1
- **NEW**: per-layer attention KL averaged over 8 paired layers, weight 0.05

**Teacher is LIVE this phase.** Phase-0 cached hidden states and top-K logits
but not attention maps (too large to cache — [B, T, T, L, H] ~ 840 TB full).
We run the teacher backbone forward on cuda:1 in inference mode with captured
attention weights.

**Memory.**
- cuda:0: student transformer + adapters (trainable, ~9.5 GB)
- cuda:1: teacher backbone + student frozen heads (~8.2 GB)

Teacher depformer is NOT loaded — Phase 3 only touches the temporal transformer.

**Gate to Phase 4:** validation hidden cosine > 0.85 (currently 0.900 from P2,
so phase 3 is quality polish rather than a gate unblock).

**Init:** `mhassann/moshi-p2-ckpt/ckpt_step_611.pt`

**Datasets required** (attach ALL before running):
- `mhassann/moshi-cache-s{0..2}p{0..3}` (12 datasets)
- `mhassann/moshi-cache-codes`
- `mhassann/moshi-p2-ckpt`
- `tasfiatanha/moshi-frozen-heads`
- `tasfiatanha/moshi-repo`
- Teacher weights dataset (see Cell 6 — downloads `moshiko_pytorch_bf16.safetensors`
  from HuggingFace at runtime if no local copy; ~15 GB)

**Session plan:** ~15k steps total across 2 sessions. Expect ~4 s/step (teacher
forward adds ~2x per-step cost vs Phase 2).


## Cell 1 — Global patches

In [1]:
import os, sys
# MUST be set BEFORE `import torch` or the CUDA allocator ignores it.
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch._dynamo.config.disable = True
# sys.stdout.reconfigure(encoding="utf-8")
print("torch.compile disabled, expandable_segments enabled")
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))


torch.compile disabled, expandable_segments enabled
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True


## Cell 2 — Environment verification

In [2]:
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

assert torch.cuda.device_count() >= 2, "Phase 3 needs dual GPU (student cuda:0, teacher cuda:1)"
torch.cuda.set_device(0)
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.10.0+cu128   cuda: 12.8
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
=== environment check PASSED ===


## Cell 3 — Installs

In [3]:
import subprocess, sys, os

for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "bitsandbytes>=0.45.0",
    "sentencepiece",
    "einops",
    "safetensors",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

MOSHI_SRC = "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
if MOSHI_DST not in sys.path:
    sys.path.insert(0, MOSHI_DST)
importlib.invalidate_caches()

import moshi, transformers, bitsandbytes
print(f"moshi from: {moshi.__file__}")
print(f"transformers: {transformers.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("=== installs OK ===")

import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.3 MB/s eta 0:00:00
moshi from: /kaggle/working/moshi_repo/moshi/__init__.py
transformers: 4.44.2
bitsandbytes: 0.49.2
=== installs OK ===
CUDAGraphed monkey-patched to no-op


## Cell 4 — Write smol_temporal.py

Same student wrapper as Phase 2. The attention-capture forward for the student
is injected later (Cell 7) as a monkey-patch, so this file is unchanged from
the Phase-2 final version.


In [4]:
import pathlib

DST = pathlib.Path('/kaggle/working/moshi_repo/moshi/models/smol_temporal.py')

_SRC = '''# moshi/models/smol_temporal.py
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
import torch
import torch.nn as nn
import transformers
from ..modules.streaming import StreamingModule, State


@dataclass
class _SmolState(State):
    past_key_values: Optional[tuple] = field(default=None)

    def reset(self, reset_mask: torch.Tensor) -> None:
        super().reset(reset_mask)
        self.past_key_values = None


class SmolTemporalTransformer(StreamingModule[_SmolState]):
    def __init__(
        self,
        teacher_dim: int = 4096,
        student_dim: int = 2048,
        hf_name: str = "HuggingFaceTB/SmolLM2-1.7B",
        rope_theta: float = 10_000.0,
        device: str = "cuda:0",
        dtype: torch.dtype = torch.float16,
    ):
        super().__init__()
        self.teacher_dim = teacher_dim
        self.student_dim = student_dim

        cfg = transformers.AutoConfig.from_pretrained(hf_name)
        cfg.rope_theta = rope_theta
        cfg.use_cache = True
        cfg.attn_implementation = "eager"
        self.backbone = transformers.AutoModel.from_pretrained(
            hf_name, config=cfg, torch_dtype=dtype,
        )
        if hasattr(self.backbone, "embed_tokens"):
            self.backbone.embed_tokens = nn.Identity()

        self.in_adapter  = nn.Linear(teacher_dim, student_dim, bias=False)
        self.out_adapter = nn.Linear(student_dim, teacher_dim, bias=False)
        nn.init.normal_(self.in_adapter.weight,  std=1.0 / (teacher_dim ** 0.5))
        nn.init.normal_(self.out_adapter.weight, std=1.0 / (student_dim ** 0.5))

        self.to(device=device, dtype=dtype)

    def _init_streaming_state(self, batch_size: int) -> _SmolState:
        device = self.in_adapter.weight.device
        return _SmolState(batch_size=batch_size, device=device, past_key_values=None)

    def forward(
        self,
        x: torch.Tensor,
        cross_attention_src: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        assert cross_attention_src is None
        assert x.dim() == 3 and x.shape[-1] == self.teacher_dim

        x = x.to(self.in_adapter.weight.device)

        past_kv = (self._streaming_state.past_key_values
                   if self._streaming_state is not None else None)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            h = self.in_adapter(x)
            out = self.backbone(
                inputs_embeds=h,
                past_key_values=past_kv,
                use_cache=(self._streaming_state is not None),
                return_dict=True,
                output_attentions=True,  # Phase-3: capture attention maps
            )
            y = self.out_adapter(out.last_hidden_state)

        if self._streaming_state is not None:
            self._streaming_state.past_key_values = out.past_key_values

        # Stash attentions on self for the training loop to pull (avoids
        # rewriting forward_text's return signature). .attentions is a tuple
        # of [B, H, T, T] tensors, one per layer.
        self._last_attentions = out.attentions

        return y

    def student_state_dict(self):
        return {
            "backbone":    self.backbone.state_dict(),
            "in_adapter":  self.in_adapter.state_dict(),
            "out_adapter": self.out_adapter.state_dict(),
        }

    def load_student_state_dict(self, sd: dict):
        self.backbone.load_state_dict(sd["backbone"])
        self.in_adapter.load_state_dict(sd["in_adapter"])
        self.out_adapter.load_state_dict(sd["out_adapter"])
'''

DST.write_text(_SRC)
print(f"Wrote {DST} ({DST.stat().st_size} bytes)")

import importlib, moshi.models
if hasattr(moshi.models, "smol_temporal"):
    importlib.reload(moshi.models.smol_temporal)
from moshi.models.smol_temporal import SmolTemporalTransformer
print(f"SmolTemporalTransformer imported OK: {SmolTemporalTransformer}")
print("=== Cell 4 PASSED ===")


Wrote /kaggle/working/moshi_repo/moshi/models/smol_temporal.py (3578 bytes)
SmolTemporalTransformer imported OK: <class 'moshi.models.smol_temporal.SmolTemporalTransformer'>
=== Cell 4 PASSED ===


## Cell 5 — Open cache memmap handles + codes

Same caches as Phase 2. We use the codes (input) + cached hidden (target #1)
+ cached topk (target #2). Attention maps (target #3) come from the live
teacher in Cell 6.


In [5]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random

WINDOWS_PER_PART = 5_000
TOTAL_WINDOWS    = 60_000
T_FRAMES         = 375
TEACHER_DIM      = 4096
N_CB             = 17
TOP_K            = 256
VAL_FRACTION     = 0.02
SEED             = 42

parts_hidden = []
parts_idx    = []
parts_val    = []

for shard in range(3):
    for part in range(4):
        for prefix in [
            f"/kaggle/input/moshi-cache-s{shard}p{part}",
            f"/kaggle/input/datasets/mhassann/moshi-cache-s{shard}p{part}",
        ]:
            import os
            if os.path.isdir(prefix):
                break
        parts_hidden.append(np.memmap(f"{prefix}/hidden.npy",
            dtype="float16", mode="r", shape=(5000, 375, 4096)))
        parts_idx.append(np.memmap(f"{prefix}/topk_idx.npy",
            dtype="int32",   mode="r", shape=(5000, 375, 256)))
        parts_val.append(np.memmap(f"{prefix}/topk_val.npy",
            dtype="float16", mode="r", shape=(5000, 375, 256)))

print(f"Opened {len(parts_hidden)} hidden + {len(parts_idx)} idx + {len(parts_val)} val memmap handles")

codes_path_candidates = [
    "/kaggle/input/moshi-cache-codes/codes.npy",
    "/kaggle/input/datasets/mhassann/moshi-cache-codes/codes.npy",
]
codes_path = None
for cp in codes_path_candidates:
    if os.path.exists(cp):
        codes_path = cp
        break
if codes_path is None:
    raise FileNotFoundError(f"codes.npy not found in: {codes_path_candidates}")

codes_mm = np.memmap(codes_path, dtype="int16", mode="r",
                     shape=(TOTAL_WINDOWS, N_CB, T_FRAMES))
print(f"Codes memmap: {codes_mm.shape} dtype={codes_mm.dtype}")

# Same seed + split as Phases 1 + 2 → resuming is safe, val never leaks into train.
rng = random.Random(SEED)
all_indices = list(range(TOTAL_WINDOWS))
rng.shuffle(all_indices)
n_val = max(1, int(TOTAL_WINDOWS * VAL_FRACTION))
val_indices = set(all_indices[:n_val])
train_indices = [i for i in range(TOTAL_WINDOWS) if i not in val_indices]
print(f"Train: {len(train_indices)}, Val: {n_val}")


class CacheDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        part_i, local_i = divmod(i, WINDOWS_PER_PART)

        h   = np.array(parts_hidden[part_i][local_i], copy=True)
        ti  = np.array(parts_idx[part_i][local_i], copy=True)
        tv  = np.array(parts_val[part_i][local_i], copy=True)
        c   = np.array(codes_mm[i], copy=True)

        return {
            "hidden":   torch.from_numpy(h),
            "topk_idx": torch.from_numpy(ti),
            "topk_val": torch.from_numpy(tv),
            "codes":    torch.from_numpy(c.astype(np.int64)),
        }


train_ds = CacheDataset(train_indices)
val_ds   = CacheDataset(list(val_indices))

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                          num_workers=0, pin_memory=True)

batch = train_ds[0]
print(f"Sample: hidden {batch['hidden'].shape} {batch['hidden'].dtype}, "
      f"codes {batch['codes'].shape} {batch['codes'].dtype}, "
      f"topk_val {batch['topk_val'].shape} {batch['topk_val'].dtype}")
print("=== Cell 5 PASSED ===")


Opened 12 hidden + 12 idx + 12 val memmap handles
Codes memmap: (60000, 17, 375) dtype=int16
Train: 58800, Val: 1200
Sample: hidden torch.Size([375, 4096]) torch.float16, codes torch.Size([17, 375]) torch.int64, topk_val torch.Size([375, 256]) torch.float16
=== Cell 5 PASSED ===


## Cell 6 — Load LIVE Moshi teacher on cuda:1

This is the Phase-3 novelty. Unlike Phases 1/2 (which trained purely from
cached targets), Phase 3 needs live attention weights, which can't be cached.

We load only the `transformer`, `emb`, `text_emb` components of Moshi —
depformer/text_linear/linears are NOT needed because we only distill up to
and through the temporal transformer.

Teacher fits in ~7 GB on cuda:1 in fp16 (32-layer, 4096-dim backbone).


In [6]:
import torch, gc, pathlib, os, ctypes
from moshi.models.loaders import CheckpointInfo
from moshi.models.lm import LMModel

# --- glibc arena trim helper (Linux) ------------------------------------
# Python `del` / `gc.collect()` only drop references; the underlying heap
# arenas in glibc are NOT returned to the OS by default. Large transient
# allocations (like loading a 16 GB fp16 teacher state_dict into CPU RAM)
# leave that memory as resident-but-free RSS — which is what killed Cell 8
# (Kaggle RSS was already 29 GB before Cell 8 even started).
#
# `malloc_trim(0)` forces glibc to release freed pages. On Kaggle (Ubuntu)
# this typically reclaims 10+ GB after a large tensor load + del. Safe no-op
# on non-glibc platforms.
try:
    _libc = ctypes.CDLL("libc.so.6")
    def _trim_ram():
        try:
            _libc.malloc_trim(0)
        except Exception:
            pass
except OSError:
    def _trim_ram():
        pass


def _host_rss_gb():
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
    except Exception:
        return -1.0

def _mem(tag):
    torch.cuda.synchronize()
    parts = [f"host={_host_rss_gb():5.1f}GB"]
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        parts.append(f"cuda:{i} {(total-free)/1e9:4.1f}/{total/1e9:4.1f}GB")
    print(f"[MEM {tag:32s}] " + "  ".join(parts))


_mem("start of Cell 6")

# Teacher is built from the same hardcoded config as the student shell, then
# its backbone weights are loaded from the Kaggle-mirrored teacher dataset.

print("Building teacher shell from hardcoded moshiko config ...")
teacher_lm = LMModel(
    dim=4096, num_heads=32, num_layers=32, hidden_scale=4.125,
    gating="silu", norm="rms_norm_f32", positional_embedding="rope",
    context=3000, n_q=16, dep_q=8, card=2048, text_card=32000,
    delays=[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    depformer_dim=1024, depformer_dim_feedforward=4224,
    depformer_num_heads=16, depformer_num_layers=6,
    depformer_multi_linear=True, depformer_weights_per_step=True,
    depformer_context=8, depformer_pos_emb="none",
    existing_text_padding_id=3,
).to(dtype=torch.float16)
_mem("after teacher shell")

# Drop the unused submodules BEFORE loading weights — saves allocating
# fp16 random-init buffers we'd throw away anyway.
for attr_to_drop in ["depformer", "depformer_in", "depformer_emb",
                     "depformer_text_emb", "linears", "text_linear", "out_norm"]:
    if hasattr(teacher_lm, attr_to_drop):
        delattr(teacher_lm, attr_to_drop)
gc.collect(); _trim_ram()
_mem("after dropping unused heads")

# Move backbone + embs to cuda:1 BEFORE loading weights. Then we stream
# safetensors keys one-at-a-time into GPU-resident parameters, so CPU RAM
# never sees the full ~8 GB teacher dict at once.
print("Moving teacher shell (backbone + embs) to cuda:1 pre-load ...")
teacher_lm.transformer.to("cuda:1")
teacher_lm.emb.to("cuda:1")
teacher_lm.text_emb.to("cuda:1")
gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("after shell -> cuda:1")

# Find teacher weights.
TEACHER_WEIGHT_CANDIDATES = [
    "/kaggle/input/datasets/tasfiatanha/kyutai-moshiko/model.safetensors",
    "/kaggle/input/kyutai-moshiko/model.safetensors",
    "/kaggle/input/datasets/tasfiatanha/moshi-teacher-weights/model.safetensors",
    "/kaggle/input/moshi-teacher-weights/model.safetensors",
]
teacher_weights_path = None
for p in TEACHER_WEIGHT_CANDIDATES:
    if os.path.exists(p):
        teacher_weights_path = p
        break

if teacher_weights_path is None:
    print("Teacher weights not found locally — downloading from HF Hub ...")
    from huggingface_hub import hf_hub_download
    teacher_weights_path = hf_hub_download(
        repo_id="kyutai/moshiko-pytorch-bf16",
        filename="model.safetensors",
        cache_dir="/tmp/moshi_teacher",
    )
    print(f"Downloaded teacher to: {teacher_weights_path}")

# ---------------------------------------------------------------------------
# Streaming load: safe_open + per-key copy_ into GPU-resident parameters.
# Never holds the whole state_dict on host — each tensor is GPU-bound as soon
# as it's read off disk. malloc_trim between keys to keep RSS flat.
# ---------------------------------------------------------------------------
from safetensors import safe_open

print(f"Streaming teacher weights from {teacher_weights_path} ...")
# Build a name → parameter map over the (already GPU-resident) shell. Only
# keys that exist here will be loaded; the rest are skipped (they belong to
# the dropped depformer/heads).
_shell_params = dict(teacher_lm.named_parameters())
_shell_buffers = dict(teacher_lm.named_buffers())

loaded = 0
skipped_keys = []
with safe_open(teacher_weights_path, framework="pt") as f:
    for k in f.keys():
        target = _shell_params.get(k)
        if target is None:
            target = _shell_buffers.get(k)
        if target is None:
            skipped_keys.append(k)
            continue
        t = f.get_tensor(k)  # loaded as CPU tensor
        # Cast + move in one shot, then immediately drop the CPU copy.
        with torch.no_grad():
            target.copy_(t.to(dtype=target.dtype, device=target.device))
        del t
        loaded += 1
        if loaded % 100 == 0:
            gc.collect(); _trim_ram()

del _shell_params, _shell_buffers
gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("after streaming teacher load")
print(f"  loaded {loaded} tensors; skipped {len(skipped_keys)} "
      f"(dropped-head keys, expected)")

# Freeze everything and set eval mode
for p in teacher_lm.parameters():
    p.requires_grad_(False)
teacher_lm.eval()

for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} free {free/1e9:.2f} / {total/1e9:.2f} GB")

n_teacher = sum(p.numel() for p in teacher_lm.parameters())
print(f"Teacher params (loaded): {n_teacher/1e6:.1f} M  ({n_teacher/1e9:.3f} B)")

# Patch the teacher's forward_text to run just emb/text_emb → transformer
# and return (transformer_out, attentions). Depformer path disabled.
import types as _types

def _teacher_trunk_forward(self, sequence):
    """Runs teacher backbone ONLY, returns hidden.
    No text_linear, no depformer. Input codes come from Mimi."""
    B, K, S = sequence.shape
    emb_device = next(self.emb[0].parameters()).device
    input_sequence = sequence.to(emb_device)

    input_ = None
    for cb_index in range(self.num_audio_codebooks):
        audio_emb = self.emb[cb_index](input_sequence[:, cb_index + self.audio_offset])
        input_ = audio_emb if input_ is None else input_ + audio_emb
    text_emb = self.text_emb(input_sequence[:, 0])
    input_ = text_emb if input_ is None else input_ + text_emb

    # Teacher transformer with attention capture (patched in Cell 7).
    transformer_out = self.transformer(input_)
    return transformer_out

teacher_lm.trunk_forward = _types.MethodType(_teacher_trunk_forward, teacher_lm)
print("teacher.trunk_forward() method added")

gc.collect(); _trim_ram()
_mem("end of Cell 6")
print("=== Cell 6 PASSED ===")


[MEM start of Cell 6                 ] host=  0.9GB  cuda:0  0.1/15.6GB  cuda:1  0.1/15.6GB
Building teacher shell from hardcoded moshiko config ...
[MEM after teacher shell             ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1  0.1/15.6GB
[MEM after dropping unused heads     ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1  0.1/15.6GB
Moving teacher shell (backbone + embs) to cuda:1 pre-load ...
[MEM after shell -> cuda:1           ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1 13.8/15.6GB
Teacher weights not found locally — downloading from HF Hub ...


model.safetensors:   0%|          | 0.00/15.4G [00:00<?, ?B/s]

Downloaded teacher to: /tmp/moshi_teacher/models--kyutai--moshiko-pytorch-bf16/snapshots/2bfc9ae6e89079a5cc7ed2a68436010d91a3d289/model.safetensors
Streaming teacher weights from /tmp/moshi_teacher/models--kyutai--moshiko-pytorch-bf16/snapshots/2bfc9ae6e89079a5cc7ed2a68436010d91a3d289/model.safetensors ...
[MEM after streaming teacher load    ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1 13.8/15.6GB
  loaded 145 tensors; skipped 210 (dropped-head keys, expected)
cuda:0 free 15.53 / 15.64 GB
cuda:1 free 1.83 / 15.64 GB
Teacher params (loaded): 6842.3 M  (6.842 B)
teacher.trunk_forward() method added
[MEM end of Cell 6                   ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1 13.8/15.6GB
=== Cell 6 PASSED ===


## Cell 7 — Patch teacher + student attention layers for weight capture

Moshi's `StreamingMultiheadAttention` uses `F.scaled_dot_product_attention`
which does not return attention weights. SmolLM2 (HF) with
`attn_implementation="eager"` + `output_attentions=True` returns them naturally.
So we only patch the TEACHER side here. The student is already set up in Cell 4
(`output_attentions=True` + `attn_implementation="eager"`).

The patch replaces teacher `StreamingMultiheadAttention.forward` with an
explicit softmax(QK^T / sqrt(d)) path that stores attn weights on the module.
Only done for teacher — no gradients needed, just numerics.


In [7]:
import math
import torch
import torch.nn.functional as F
from einops import rearrange
from moshi.modules.transformer import StreamingMultiheadAttention


def _attn_capture_forward(self, query, key, value):
    """Teacher-side replacement for StreamingMultiheadAttention.forward.
    Computes attention weights explicitly (no SDPA) so they can be captured.
    Stores last attention on self._last_attn_weights ([B, H, T, T]).
    """
    state = self._streaming_state
    B, T = query.shape[:2]

    if state is None:
        offset = torch.zeros(B, device=query.device, dtype=torch.long)
        offset_cpu = 0
    else:
        offset = state.offset
        offset_cpu = state.offset_cpu

    from moshi.modules.transformer import apply_weights_per_step, expand_repeated_kv

    assert not self.cross_attention, "Phase 3 only distills self-attention layers"
    projected = apply_weights_per_step(
        self.in_projs, self.weights_per_step_schedule, query, offset_cpu)

    if self.kv_repeat == 1:
        q, k, v = rearrange(
            projected, "b t (p h d) -> p b h t d", p=3, h=self.num_heads
        )
    else:
        q = rearrange(projected[:, :, :self.embed_dim], "b t (h d) -> b h t d", h=self.num_heads)
        k, v = rearrange(
            projected[:, :, self.embed_dim:],
            "b t (p kh d) -> p b kh t d", p=2, kh=self.num_heads // self.kv_repeat
        )

    if self.rope:
        q, k = self.rope(q, k, offset, time_before_heads=False)

    k, v, pos_k = self._complete_kv(k, v)
    if self.kv_repeat > 1:
        k = expand_repeated_kv(k, self.kv_repeat)
        v = expand_repeated_kv(v, self.kv_repeat)

    pos_k = pos_k[:, None]
    if self.causal:
        pos_q = offset.view(-1, 1, 1) + torch.arange(T, device=q.device, dtype=torch.long).view(-1, 1)
        delta = pos_q - pos_k
        attn_mask_bool = (pos_k >= 0) & (delta >= 0)
        if self.context is not None:
            attn_mask_bool = attn_mask_bool & (delta < self.context)
        attn_mask_bool = attn_mask_bool[:, None]  # [B, 1, T, T_k]
    else:
        attn_mask_bool = None

    # Explicit attention computation (instead of SDPA) so we can capture weights
    d_k = q.shape[-1]
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)  # [B, H, T, T_k]

    if attn_mask_bool is not None:
        scores = scores.masked_fill(~attn_mask_bool, float("-inf"))

    attn_weights = F.softmax(scores, dim=-1)  # [B, H, T, T_k]
    # Handle NaN from all-masked rows (shouldn't happen with causal, but safe)
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

    # STORE for training loop pickup
    self._last_attn_weights = attn_weights.detach()  # teacher side, no grad

    x = torch.matmul(attn_weights, v)  # [B, H, T, d_k]
    x = rearrange(x, "b h t d -> b t (h d)")
    x = apply_weights_per_step(
        self.out_projs, self.weights_per_step_schedule, x, offset_cpu)

    if state is not None and not self.cross_attention:
        state.offset[:] = torch.where(
            state.exec_mask, state.offset + T, state.offset)
        state.offset_cpu += T
    return x


# Apply patch ONLY to teacher's self-attention modules. Do NOT patch globally —
# that would affect student's (irrelevant, since student uses HF) and any other
# StreamingMultiheadAttention instances unexpectedly.
patched_count = 0
for layer in teacher_lm.transformer.layers:
    sa = layer.self_attn
    # Bind the new forward to THIS specific instance
    import types as _types
    sa.forward = _types.MethodType(_attn_capture_forward, sa)
    patched_count += 1
print(f"Patched {patched_count} teacher self-attention layers "
      f"(expected {teacher_lm.transformer.layers.__len__()})")
print("=== Cell 7 PASSED ===")


Patched 32 teacher self-attention layers (expected 32)
=== Cell 7 PASSED ===


## Cell 8 — Build student (same as Phase 2)

Student backbone + adapters on cuda:0 (trainable, fp16). Frozen heads on cuda:1
(sharing the GPU with the teacher). Forward_text is the same cross-device patch.


In [8]:
  import gc, sys, torch
  gc.collect(); torch.cuda.empty_cache()

  # Find top CPU-resident tensors by size
  import torch
  cpu_tensors = []
  for obj in gc.get_objects():
      try:
          if torch.is_tensor(obj) and obj.device.type == "cpu":
              cpu_tensors.append((obj.numel() * obj.element_size(), tuple(obj.shape), obj.dtype))
      except Exception:
          pass
  cpu_tensors.sort(reverse=True)
  total = sum(sz for sz,_,_ in cpu_tensors)
  print(f"Total CPU tensor bytes: {total/1e9:.2f} GB across {len(cpu_tensors)} tensors")
  for sz, shape, dt in cpu_tensors[:15]:
      print(f"  {sz/1e6:8.1f} MB  {dt}  {shape}")

  # Also show host RSS
  try:
      import resource
      print(f"RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024/1024:.1f} GB")
  except Exception:
      pass

Total CPU tensor bytes: 0.00 GB across 6 tensors
       3.1 MB  torch.float16  (375, 4096)
       0.4 MB  torch.int32  (375, 256)
       0.2 MB  torch.float16  (375, 256)
       0.1 MB  torch.int64  (17, 375)
       0.0 MB  torch.float32  (16,)
       0.0 MB  torch.float32  (16,)
RSS: 29.5 GB


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [9]:
import torch, pathlib, gc, ctypes
import pickle as _pickle

# Same glibc arena-trim trick as Cell 6 — essential on Kaggle to avoid
# session kill from transient CPU-RAM spikes that don't return to the OS.
try:
    _libc = ctypes.CDLL("libc.so.6")
    def _trim_ram():
        try:
            _libc.malloc_trim(0)
        except Exception:
            pass
except OSError:
    def _trim_ram():
        pass

# numpy 2.x pickle compat (same as Phase 2)
class _NumpyCompatUnpickler(_pickle.Unpickler):
    _REMAP = {
        "numpy.core.multiarray": "numpy._core.multiarray",
        "numpy.core.numeric":    "numpy._core.numeric",
        "numpy.core.umath":      "numpy._core.umath",
        "numpy.core":            "numpy._core",
    }
    def find_class(self, module, name):
        return super().find_class(self._REMAP.get(module, module), name)

class _NpPickle:
    Unpickler        = _NumpyCompatUnpickler
    loads            = staticmethod(_pickle.loads)
    load             = staticmethod(_pickle.load)
    dump             = staticmethod(_pickle.dump)
    dumps            = staticmethod(_pickle.dumps)
    HIGHEST_PROTOCOL = _pickle.HIGHEST_PROTOCOL
    DEFAULT_PROTOCOL = _pickle.DEFAULT_PROTOCOL
    PickleError      = _pickle.PickleError
    UnpicklingError  = _pickle.UnpicklingError

def _torch_load(path, **kw):
    kw.setdefault("weights_only", False)
    kw.setdefault("pickle_module", _NpPickle)
    return torch.load(path, **kw)


def _host_rss_gb():
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
    except Exception:
        return -1.0

def _mem(tag):
    torch.cuda.synchronize()
    parts = [f"host={_host_rss_gb():5.1f}GB"]
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        parts.append(f"cuda:{i} {(total-free)/1e9:4.1f}/{total/1e9:4.1f}GB")
    print(f"[MEM {tag:32s}] " + "  ".join(parts))


# Trim arenas first — if Cell 6 (rewritten) was still over budget, this
# reclaims the slack before we start allocating the student.
gc.collect(); _trim_ram()
_mem("start of Cell 8")

from moshi.models.lm import LMModel

print("Building student shell ...")
student_lm = LMModel(
    dim=4096, num_heads=32, num_layers=32, hidden_scale=4.125,
    gating="silu", norm="rms_norm_f32", positional_embedding="rope",
    context=3000, n_q=16, dep_q=8, card=2048, text_card=32000,
    delays=[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    depformer_dim=1024, depformer_dim_feedforward=4224,
    depformer_num_heads=16, depformer_num_layers=6,
    depformer_multi_linear=True, depformer_weights_per_step=True,
    depformer_context=8, depformer_pos_emb="none",
    existing_text_padding_id=3,
).to(dtype=torch.float16)
_mem("after LMModel shell (CPU)")

# Move heads to their target GPUs BEFORE replacing the TT. That frees the
# CPU copies of these modules immediately (each .to() on a submodule creates
# a new storage on GPU and releases the CPU one once GC runs).
STUDENT_HEADS_CUDA0 = ["depformer", "depformer_emb", "depformer_text_emb"]
STUDENT_HEADS_CUDA1 = ["emb", "text_emb", "out_norm", "text_linear",
                       "depformer_in", "linears"]
for attr in STUDENT_HEADS_CUDA0:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:0")
for attr in STUDENT_HEADS_CUDA1:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:1")
gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("after heads -> split GPUs")

print("Replacing Helium TT with SmolLM2-1.7B (loaded directly on cuda:0) ...")
# SmolLM2's HF loader pulls safetensors directly into cuda:0 tensors because
# we pass device="cuda:0" in the SmolTemporalTransformer ctor. No CPU fp16
# copy of the 3.5 GB backbone is materialized.
smol_tt = SmolTemporalTransformer(
    teacher_dim=4096, student_dim=2048,
    hf_name="HuggingFaceTB/SmolLM2-1.7B",
    rope_theta=10_000.0,
    device="cuda:0", dtype=torch.float16,
)
# Drop the old Helium TT (still on CPU from the shell construction).
old_tt = student_lm.transformer
student_lm.transformer = smol_tt
del old_tt
gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("after smol_tt swap (old TT dropped)")

# Requires_grad: only the new TT is trainable.
for name, p in student_lm.named_parameters():
    p.requires_grad_(name.startswith("transformer."))

# Frozen heads — load each .pt directly onto its FINAL GPU.
FROZEN_CANDIDATES = [
    pathlib.Path("/kaggle/input/datasets/tasfiatanha/moshi-frozen-heads"),
    pathlib.Path("/kaggle/input/moshi-frozen-heads"),
]
frozen_dir = None
for p in FROZEN_CANDIDATES:
    if p.exists():
        frozen_dir = p
        break
if frozen_dir is None:
    raise FileNotFoundError(f"Frozen heads not found: {FROZEN_CANDIDATES}")

print(f"Loading frozen heads from {frozen_dir} (direct-to-GPU) ...")
_head_device = {
    "emb": "cuda:1", "text_emb": "cuda:1", "out_norm": "cuda:1",
    "text_linear": "cuda:1", "depformer_in": "cuda:1", "linears": "cuda:1",
    "depformer": "cuda:0", "depformer_emb": "cuda:0",
    "depformer_text_emb": "cuda:0",
}
for name, dev in _head_device.items():
    pt_file = frozen_dir / f"{name}.pt"
    if not pt_file.exists():
        continue
    sd = torch.load(pt_file, map_location=dev, weights_only=True)
    getattr(student_lm, name).load_state_dict(sd)
    del sd
    gc.collect(); _trim_ram()
_mem("after frozen heads loaded")

print("Keeping trainables in fp16 (bnb handles fp32 master)")

n_train = sum(p.numel() for p in student_lm.parameters() if p.requires_grad)
n_froz  = sum(p.numel() for p in student_lm.parameters() if not p.requires_grad)
print(f"Trainable : {n_train/1e6:.1f} M  ({n_train/1e9:.3f} B)")
print(f"Frozen    : {n_froz/1e6:.1f} M  ({n_froz/1e9:.3f} B)")

# Cross-device forward_text patch: emb lives on cuda:1, TT on cuda:0,
# text_linear on cuda:1 — we hop accordingly.
import types as _types

def _fixed_forward_text(self, sequence, sum_condition=None, cross_attention_src=None):
    B, K, S = sequence.shape
    assert K == self.num_codebooks

    emb_device = next(self.emb[0].parameters()).device
    input_sequence = sequence.to(emb_device)

    input_ = None
    for cb_index in range(self.num_audio_codebooks):
        audio_emb = self.emb[cb_index](input_sequence[:, cb_index + self.audio_offset])
        input_ = audio_emb if input_ is None else input_ + audio_emb
    text_emb = self.text_emb(input_sequence[:, 0])
    input_ = text_emb if input_ is None else input_ + text_emb

    if sum_condition is not None:
        input_ = input_ + sum_condition.to(input_)
    if cross_attention_src is not None:
        cross_attention_src = cross_attention_src.to(input_)

    tt_device = next(self.transformer.parameters()).device
    transformer_out = self.transformer(
        input_.to(tt_device), cross_attention_src=cross_attention_src,
    )

    if self.out_norm:
        on_device = next(self.out_norm.parameters()).device
        transformer_out = self.out_norm(transformer_out.to(on_device))

    tl_device = next(self.text_linear.parameters()).device
    text_logits = self.text_linear(transformer_out.to(tl_device))
    text_logits = text_logits[:, None]
    return transformer_out, text_logits

student_lm.forward_text = _types.MethodType(_fixed_forward_text, student_lm)

gc.collect(); torch.cuda.empty_cache(); _trim_ram()
_mem("end of Cell 8")
print("forward_text patched (cross-device, student-split layout)")
print("=== Cell 8 PASSED ===")


[MEM start of Cell 8                 ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1 13.8/15.6GB
Building student shell ...
[MEM after LMModel shell (CPU)       ] host= 29.5GB  cuda:0  0.1/15.6GB  cuda:1 13.8/15.6GB
[MEM after heads -> split GPUs       ] host= 29.5GB  cuda:0  1.5/15.6GB  cuda:1 14.7/15.6GB
Replacing Helium TT with SmolLM2-1.7B (loaded directly on cuda:0) ...


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

[MEM after smol_tt swap (old TT dropped)] host= 29.5GB  cuda:0  4.7/15.6GB  cuda:1 14.7/15.6GB
Loading frozen heads from /kaggle/input/datasets/tasfiatanha/moshi-frozen-heads (direct-to-GPU) ...
[MEM after frozen heads loaded       ] host= 29.5GB  cuda:0  5.9/15.6GB  cuda:1 15.0/15.6GB
Keeping trainables in fp16 (bnb handles fp32 master)
Trainable : 1627.5 M  (1.627 B)
Frozen    : 1110.8 M  (1.111 B)
[MEM end of Cell 8                   ] host= 29.5GB  cuda:0  4.7/15.6GB  cuda:1 14.7/15.6GB
forward_text patched (cross-device, student-split layout)
=== Cell 8 PASSED ===


## Cell 9 — Optimizer, scheduler, gradient checkpointing

Phase-3 LR is even lower than Phase-2 (2e-5 vs 5e-5). Quality polish.


In [10]:
import bitsandbytes as bnb
import math

GRAD_ACCUM   = 4
LR           = 2e-5        # lower than Phase-2 (5e-5): quality polish
LR_MIN       = 2e-6
WARMUP_STEPS = 200
MAX_STEPS    = 6_000       # per session; gate already met (cos>0.85)
MAX_NORM     = 5.0

trainable_params = [p for p in student_lm.parameters() if p.requires_grad]
assert len(trainable_params) > 0, "No trainable params!"

optimizer = bnb.optim.PagedAdamW8bit(trainable_params, lr=LR, weight_decay=0.01)

class _NoOpScaler:
    def scale(self, loss):   return loss
    def unscale_(self, opt): pass
    def step(self, opt):     opt.step()
    def update(self):        pass
    def get_scale(self):     return 1
    def state_dict(self):    return {}
    def load_state_dict(self, sd): pass
scaler = _NoOpScaler()

backbone = student_lm.transformer.backbone
backbone.config.use_cache = False
if hasattr(backbone, "gradient_checkpointing_enable"):
    backbone.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    gc_on = getattr(backbone, "is_gradient_checkpointing", False)
    print(f"Gradient checkpointing: is_gradient_checkpointing={gc_on}, "
          f"use_cache={backbone.config.use_cache}")
student_lm.train()
backbone.train()

def lr_schedule(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return max(LR_MIN / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

print(f"Optimizer: PagedAdamW8bit, lr={LR}, wd=0.01")
print(f"Scheduler: linear warmup {WARMUP_STEPS} steps, cosine -> {LR_MIN}")
print(f"Grad accum: {GRAD_ACCUM}, max_norm: {MAX_NORM}")
print(f"Max steps this session: {MAX_STEPS}")
print("=== Cell 9 PASSED ===")


Gradient checkpointing: is_gradient_checkpointing=True, use_cache=False
Optimizer: PagedAdamW8bit, lr=2e-05, wd=0.01
Scheduler: linear warmup 200 steps, cosine -> 2e-06
Grad accum: 4, max_norm: 5.0
Max steps this session: 6000
=== Cell 9 PASSED ===


## Cell 10 — Resume from Phase-2 checkpoint (or latest Phase-3 checkpoint)

Priority order:
1. `/kaggle/working/ckpt_step_*.pt`       (in-session Phase-3 checkpoints)
2. `/kaggle/input/.../moshi-p3-ckpt/`     (previous Phase-3 session)
3. `/kaggle/input/.../moshi-p2-ckpt/`     (fresh start from Phase-2 final)

When resuming from Phase-2: loads ONLY student weights, fresh optimizer state.
When resuming from Phase-3: loads weights + optimizer + scheduler + RNG.


In [11]:
import pathlib, glob, json, random, os
import numpy as np

P3_CANDIDATES = [
    "/kaggle/working",
    "/kaggle/input/datasets/mhassann/moshi-p3-ckpt",
    "/kaggle/input/moshi-p3-ckpt",
]
P2_CANDIDATES = [
    "/kaggle/input/datasets/mhassann/moshi-p2-ckpt",
    "/kaggle/input/moshi-p2-ckpt",
]

start_step = 0
total_wall = 0.0
ckpt_loaded = False
ckpt_phase  = None

for ckpt_dir in P3_CANDIDATES:
    if not os.path.isdir(ckpt_dir):
        continue
    ckpts = sorted(glob.glob(f"{ckpt_dir}/ckpt_step_*.pt"))
    if not ckpts:
        continue
    latest = ckpts[-1]
    print(f"Found Phase-3 checkpoint: {latest}")
    ckpt = _torch_load(latest, map_location="cpu")
    if ckpt.get("phase") != "P3":
        print(f"  skipping — phase={ckpt.get('phase')} (not P3)")
        del ckpt
        continue

    smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
    smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
    smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
    print("  student weights restored")

    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    print("  optimizer + scheduler + scaler restored")

    if "torch_rng" in ckpt:  torch.set_rng_state(ckpt["torch_rng"])
    if "cuda_rng" in ckpt:   torch.cuda.set_rng_state_all(ckpt["cuda_rng"])
    if "numpy_rng" in ckpt:  np.random.set_state(ckpt["numpy_rng"])
    if "python_rng" in ckpt: random.setstate(ckpt["python_rng"])

    start_step = ckpt.get("step", 0)
    total_wall = ckpt.get("wall_seconds", 0.0)
    ckpt_loaded = True
    ckpt_phase = "P3"
    del ckpt
    torch.cuda.empty_cache()
    print(f"  Resuming Phase-3 from step {start_step}")
    break

if not ckpt_loaded:
    for ckpt_dir in P2_CANDIDATES:
        if not os.path.isdir(ckpt_dir):
            continue
        ckpts = sorted(glob.glob(f"{ckpt_dir}/ckpt_step_*.pt"))
        if not ckpts:
            continue
        latest = ckpts[-1]
        print(f"Initializing from Phase-2 checkpoint: {latest}")
        ckpt = _torch_load(latest, map_location="cpu")

        smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
        smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
        smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
        print("  Phase-2 student weights loaded (fresh optimizer state)")
        ckpt_loaded = True
        ckpt_phase = "P2-init"
        del ckpt
        torch.cuda.empty_cache()
        break

if not ckpt_loaded:
    raise RuntimeError(
        "No checkpoint found — Phase 3 requires either a Phase-2 final "
        "checkpoint or a prior Phase-3 checkpoint."
    )

print(f"Start step: {start_step}, phase init: {ckpt_phase}")
print("=== Cell 10 PASSED ===")


Initializing from Phase-2 checkpoint: /kaggle/input/datasets/mhassann/moshi-p2-ckpt/ckpt_step_611.pt
  Phase-2 student weights loaded (fresh optimizer state)
Start step: 0, phase init: P2-init
=== Cell 10 PASSED ===


## Cell 11 — Loss functions (hidden cos + text JSD + attention KL)

**Attention KL.**
For each paired (student, teacher) layer we have:
  student_attn: [B, H, T, T]  fp32 probabilities  (from HF output_attentions)
  teacher_attn: [B, H, T, T]  fp32 probabilities  (from patched capture)

We compute symmetric per-head KL and average over heads, time, pairs.

Layer pairing: student layer `s` ↔ teacher layer `floor(s * 32/24)`.

To keep memory under control for large T, we:
  - downcast both to fp32 for the math (needed — fp16 softmax-KL explodes)
  - average over heads first (reduces memory by 32×)
  - KL is computed frame-by-frame using `sum over key positions`

This keeps attn_kl memory at O(B * T_chunk) instead of O(B * H * T * T).


In [12]:
import torch
import torch.nn.functional as F

ALPHA_HIDDEN = 1.0
ALPHA_TEXT   = 0.1
ALPHA_ATTN   = 0.05
VOCAB_SIZE   = 32000

# Layer pairing: student [0..23] -> teacher [floor(s * 32/24)]
STUDENT_LAYERS = 24
TEACHER_LAYERS = 32
LAYER_PAIRS = []
# Sample 8 evenly-spaced student layers to keep attn loss memory manageable.
# Map each to its nearest teacher layer.
for s in [2, 5, 8, 11, 14, 17, 20, 23]:
    t = min(TEACHER_LAYERS - 1, int(round(s * TEACHER_LAYERS / STUDENT_LAYERS)))
    LAYER_PAIRS.append((s, t))
print(f"Layer pairs (student, teacher): {LAYER_PAIRS}")


def cosine_loss(student_h, teacher_h):
    s = student_h.float()
    t = teacher_h.float().to(s.device)
    cos = F.cosine_similarity(s, t, dim=-1)
    return (1.0 - cos).mean()


def sparse_text_jsd(student_logits, topk_idx, topk_val):
    d = student_logits.device
    idx = topk_idx.long().to(d)
    val = topk_val.float().to(d)
    s_topk = torch.gather(student_logits.float(), 2, idx)

    p_student = F.softmax(s_topk, dim=-1)
    p_teacher = F.softmax(val,    dim=-1)
    log_p_s   = F.log_softmax(s_topk, dim=-1)
    log_p_t   = F.log_softmax(val,    dim=-1)

    m     = 0.5 * (p_student + p_teacher)
    log_m = m.clamp(min=1e-10).log()

    kl_pm = (p_student * (log_p_s - log_m)).sum(dim=-1).mean()
    kl_qm = (p_teacher * (log_p_t - log_m)).sum(dim=-1).mean()
    return 0.5 * (kl_pm + kl_qm)


def sparse_text_kl(student_logits, topk_idx, topk_val):
    d = student_logits.device
    idx = topk_idx.long().to(d)
    val = topk_val.float().to(d)
    s_topk = torch.gather(student_logits.float(), 2, idx)
    log_p_student = F.log_softmax(s_topk, dim=-1)
    log_p_teacher = F.log_softmax(val,    dim=-1)
    p_teacher     = F.softmax(val, dim=-1)
    kl = (p_teacher * (log_p_teacher - log_p_student)).sum(dim=-1)
    return kl.mean()


def attn_kl_loss(student_attn_list, teacher_attn_list, layer_pairs):
    """Symmetric attention-map KL over paired layers.

    student_attn_list: tuple of [B, H, T, T] from HF output_attentions
    teacher_attn_list: list of [B, H, T, T] captured from teacher patch
    layer_pairs: list of (s, t) indices

    Both are probabilities (post-softmax). We compute head-averaged symmetric
    KL per pair, then average over pairs.
    """
    losses = []
    for (s, t) in layer_pairs:
        if s >= len(student_attn_list):
            continue
        if t >= len(teacher_attn_list):
            continue
        s_attn = student_attn_list[s]
        t_attn = teacher_attn_list[t].to(s_attn.device)

        # Defensive cast + clamp (fp16 prob distributions can have zeros that
        # blow up log)
        s_p = s_attn.float().clamp(min=1e-8)
        t_p = t_attn.float().clamp(min=1e-8)
        # Renormalize after clamp
        s_p = s_p / s_p.sum(dim=-1, keepdim=True)
        t_p = t_p / t_p.sum(dim=-1, keepdim=True)

        # Symmetric KL: 0.5*(KL(s||t) + KL(t||s)) averaged over heads+time
        kl_st = (s_p * (s_p.log() - t_p.log())).sum(dim=-1)  # [B, H, T]
        kl_ts = (t_p * (t_p.log() - s_p.log())).sum(dim=-1)
        sym = 0.5 * (kl_st + kl_ts)
        losses.append(sym.mean())

    if not losses:
        return torch.tensor(0.0, device=student_attn_list[0].device)
    return torch.stack(losses).mean()


print(f"Loss functions defined")
print(f"ALPHA_HIDDEN={ALPHA_HIDDEN}, ALPHA_TEXT={ALPHA_TEXT}, ALPHA_ATTN={ALPHA_ATTN}")
print("=== Cell 11 PASSED ===")


Layer pairs (student, teacher): [(2, 3), (5, 7), (8, 11), (11, 15), (14, 19), (17, 23), (20, 27), (23, 31)]
Loss functions defined
ALPHA_HIDDEN=1.0, ALPHA_TEXT=0.1, ALPHA_ATTN=0.05
=== Cell 11 PASSED ===


## Cell 12 — Validation function

Reports val_cos_sim (P1 gate), val_text_kl (P2 gate) and val_attn_kl (P3
quality metric). The P3 GATE to Phase-4 is val_cos_sim > 0.85, already met
by Phase 2 exit.


In [13]:
@torch.no_grad()
def validate(student_model, teacher_model, val_loader, device="cuda:0"):
    student_model.eval()
    teacher_model.eval()
    total_1mc = 0.0
    total_kl  = 0.0
    total_akl = 0.0
    n = 0
    for batch in val_loader:
        codes_b    = batch["codes"].to(device)
        hidden_t   = batch["hidden"].to("cuda:1")
        topk_idx_b = batch["topk_idx"].to("cuda:1")
        topk_val_b = batch["topk_val"].to("cuda:1")

        T_total = codes_b.shape[-1]
        T_CHUNK = 125
        n_chunks = (T_total + T_CHUNK - 1) // T_CHUNK
        chunk_1mc = 0.0
        chunk_kl  = 0.0
        chunk_akl = 0.0
        for c_i in range(n_chunks):
            a, b = c_i * T_CHUNK, min((c_i + 1) * T_CHUNK, T_total)
            codes_c  = codes_b[..., a:b]
            hidden_c = hidden_t[:, a:b]
            idx_c    = topk_idx_b[:, a:b]
            val_c    = topk_val_b[:, a:b]

            # Student forward
            transformer_out, text_logits = student_model.forward_text(codes_c)
            student_attn = student_model.transformer._last_attentions

            # Teacher forward (no grad)
            _ = teacher_model.trunk_forward(codes_c)
            teacher_attn = [L.self_attn._last_attn_weights
                            for L in teacher_model.transformer.layers]

            cos = F.cosine_similarity(transformer_out.float(),
                                      hidden_c.float().to(transformer_out.device),
                                      dim=-1)
            chunk_1mc += (1.0 - cos).mean().item() * (b - a) / T_total
            chunk_kl  += sparse_text_kl(text_logits[:, 0], idx_c, val_c).item() * (b - a) / T_total
            chunk_akl += attn_kl_loss(student_attn, teacher_attn, LAYER_PAIRS).item() * (b - a) / T_total

            # Free teacher attention buffers between chunks (8×32×125×125 * fp32 = ~15 MB each, 32 layers)
            for L in teacher_model.transformer.layers:
                L.self_attn._last_attn_weights = None

        total_1mc += chunk_1mc
        total_kl  += chunk_kl
        total_akl += chunk_akl
        n += 1

    student_model.train()
    return {
        "val_cos_sim":       1.0 - total_1mc / n,
        "val_1_minus_cos":   total_1mc / n,
        "val_text_kl":       total_kl  / n,
        "val_attn_kl":       total_akl / n,
    }


print("validate() defined")
print("=== Cell 12 PASSED ===")


validate() defined
=== Cell 12 PASSED ===


## Cell 13 — Training loop

Per-chunk: student forward → teacher forward (no grad) → sum of 3 losses → backward.
Teacher forward uses `@torch.no_grad()` so its memory is reclaimed after each chunk.

Checkpoint every 1000 steps, validate every 500 steps.
Gate-check both P3 (cos>0.85) and a P3-quality signal (attn_kl drop).


In [14]:
import time, json, pathlib

OUT_DIR = pathlib.Path("/kaggle/working")
LOG_PATH = OUT_DIR / "train_log.jsonl"

student_lm.train()
teacher_lm.eval()
device = "cuda:0"

log_file = open(LOG_PATH, "a")

step = start_step
micro_step = 0
accum_loss_h = 0.0
accum_loss_j = 0.0
accum_loss_a = 0.0
t_session = time.time()

print(f"Starting Phase-3 training from step {step}, max {MAX_STEPS} steps this session")
print(f"Grad accum = {GRAD_ACCUM}")
print(f"Loss: {ALPHA_HIDDEN}*hidden_cos + {ALPHA_TEXT}*text_jsd + {ALPHA_ATTN}*attn_kl")
print()

optimizer.zero_grad(set_to_none=True)

data_iter = iter(train_loader)

while step < start_step + MAX_STEPS:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)

    codes_b    = batch["codes"].to(device)
    hidden_t   = batch["hidden"].to("cuda:1")
    topk_idx_b = batch["topk_idx"].to("cuda:1")
    topk_val_b = batch["topk_val"].to("cuda:1")

    T = codes_b.shape[-1]
    T_CHUNK = 125
    micro_h = 0.0
    micro_j = 0.0
    micro_a = 0.0
    n_chunks = (T + T_CHUNK - 1) // T_CHUNK
    for c_i in range(n_chunks):
        a, b = c_i * T_CHUNK, min((c_i + 1) * T_CHUNK, T)
        codes_c  = codes_b[..., a:b]
        hidden_c = hidden_t[:, a:b]
        idx_c    = topk_idx_b[:, a:b]
        val_c    = topk_val_b[:, a:b]

        # Student forward (trainable)
        transformer_out, text_logits = student_lm.forward_text(codes_c)
        student_attn = student_lm.transformer._last_attentions

        # Teacher forward (no grad)
        with torch.no_grad():
            _ = teacher_lm.trunk_forward(codes_c)
            teacher_attn = [L.self_attn._last_attn_weights
                            for L in teacher_lm.transformer.layers]

        # Losses (each loss ends up on its own device; bring all to cuda:0 before summing)
        loss_h = cosine_loss(transformer_out, hidden_c)
        loss_j = sparse_text_jsd(text_logits[:, 0], idx_c, val_c)
        loss_a = attn_kl_loss(student_attn, teacher_attn, LAYER_PAIRS)

        chunk_frac = (b - a) / T
        loss = (ALPHA_HIDDEN * loss_h.to(device)
                + ALPHA_TEXT * loss_j.to(device)
                + ALPHA_ATTN * loss_a.to(device)) * chunk_frac / GRAD_ACCUM
        scaler.scale(loss).backward()
        micro_h += loss_h.item() * chunk_frac
        micro_j += loss_j.item() * chunk_frac
        micro_a += loss_a.item() * chunk_frac

        # Free teacher attention between chunks
        for L in teacher_lm.transformer.layers:
            L.self_attn._last_attn_weights = None

    accum_loss_h += micro_h
    accum_loss_j += micro_j
    accum_loss_a += micro_a
    micro_step += 1

    if micro_step % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=MAX_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        step += 1
        wall = time.time() - t_session + total_wall
        avg_h = accum_loss_h / GRAD_ACCUM
        avg_j = accum_loss_j / GRAD_ACCUM
        avg_a = accum_loss_a / GRAD_ACCUM

        gn = float(grad_norm)
        gn_str = f"{gn:.3f}" if gn < 1e6 else "inf"

        row = {
            "step": step,
            "phase": "P3",
            "loss_hidden": round(avg_h, 5),
            "loss_text_jsd": round(avg_j, 5),
            "loss_attn_kl": round(avg_a, 5),
            "grad_norm": round(gn, 4) if gn < 1e6 else None,
            "lr": round(scheduler.get_last_lr()[0], 7),
            "wall_s": round(wall, 1),
        }
        log_file.write(json.dumps(row) + "\n")
        log_file.flush()

        if step % 50 == 0:
            print(f"step {step:5d}  hid={avg_h:.4f}  jsd={avg_j:.4f}  "
                  f"akl={avg_a:.4f}  gn={gn_str}  lr={row['lr']:.2e}  wall={wall:.0f}s")

        accum_loss_h = 0.0
        accum_loss_j = 0.0
        accum_loss_a = 0.0

        if step % 500 == 0:
            val_result = validate(student_lm, teacher_lm, val_loader, device)
            print(f"  VAL step {step}: cos_sim={val_result['val_cos_sim']:.4f}  "
                  f"text_kl={val_result['val_text_kl']:.4f}  "
                  f"attn_kl={val_result['val_attn_kl']:.4f}")
            val_row = {"step": step, "phase": "P3", "type": "val",
                       **val_result, "wall_s": round(wall, 1)}
            log_file.write(json.dumps(val_row) + "\n")
            log_file.flush()
            student_lm.train()

            if val_result["val_cos_sim"] > 0.85:
                print(f"  *** Phase-4 gate MET: val_cos_sim = {val_result['val_cos_sim']:.4f} > 0.85 ***")
                print("  (Note: gate was already met from Phase 2 exit. Continuing for quality polish.)")

            if val_result["val_1_minus_cos"] > 0.25:
                print(f"  ⚠ Hidden loss drifted: val_1_minus_cos = {val_result['val_1_minus_cos']:.4f} "
                      f"(P1 baseline was 0.10). Consider raising ALPHA_HIDDEN.")

        if step % 1000 == 0:
            ckpt_path = OUT_DIR / f"ckpt_step_{step}.pt"
            ckpt = {
                "step": step,
                "phase": "P3",
                "wall_seconds": round(wall, 1),
                "student_backbone": smol_tt.backbone.state_dict(),
                "in_adapter": smol_tt.in_adapter.state_dict(),
                "out_adapter": smol_tt.out_adapter.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all(),
                "numpy_rng": np.random.get_state(),
                "python_rng": random.getstate(),
                "torch_version": torch.__version__,
            }
            torch.save(ckpt, ckpt_path)
            size_gb = ckpt_path.stat().st_size / 1e9
            print(f"  CKPT saved: {ckpt_path.name} ({size_gb:.2f} GB)")
            del ckpt

log_file.close()

final_wall = time.time() - t_session + total_wall
print(f"\nTraining done. Final step: {step}, wall: {final_wall:.0f}s")

for i in range(2):
    free, total_ = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: free {free/1e9:.2f} / {total_/1e9:.2f} GB")

val_final = validate(student_lm, teacher_lm, val_loader, device)
print(f"Final VAL: cos_sim={val_final['val_cos_sim']:.4f}  "
      f"text_kl={val_final['val_text_kl']:.4f}  "
      f"attn_kl={val_final['val_attn_kl']:.4f}")
print("=== Cell 13 PASSED ===")


Starting Phase-3 training from step 0, max 6000 steps this session
Grad accum = 4
Loss: 1.0*hidden_cos + 0.1*text_jsd + 0.05*attn_kl



LlamaModel is using LlamaSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


step    50  hid=0.0830  jsd=0.0309  akl=2.3844  gn=0.032  lr=5.00e-06  wall=213s
step   100  hid=0.1080  jsd=0.0351  akl=2.3793  gn=0.045  lr=1.00e-05  wall=421s
step   150  hid=0.1019  jsd=0.0367  akl=2.3860  gn=0.036  lr=1.50e-05  wall=629s
step   200  hid=0.0884  jsd=0.0345  akl=2.3816  gn=0.035  lr=2.00e-05  wall=836s
step   250  hid=0.1100  jsd=0.0363  akl=2.3903  gn=0.038  lr=2.00e-05  wall=1043s
step   300  hid=0.0783  jsd=0.0361  akl=2.3833  gn=0.033  lr=2.00e-05  wall=1251s
step   350  hid=0.0967  jsd=0.0320  akl=2.3956  gn=0.034  lr=2.00e-05  wall=1458s
step   400  hid=0.1058  jsd=0.0399  akl=2.3821  gn=0.036  lr=1.99e-05  wall=1663s
step   450  hid=0.0911  jsd=0.0318  akl=2.3802  gn=0.033  lr=1.99e-05  wall=1869s
step   500  hid=0.1106  jsd=0.0433  akl=2.3880  gn=0.045  lr=1.99e-05  wall=2073s
  VAL step 500: cos_sim=0.9004  text_kl=0.1481  attn_kl=2.3848
  *** Phase-4 gate MET: val_cos_sim = 0.9004 > 0.85 ***
  (Note: gate was already met from Phase 2 exit. Continuing for q

KeyboardInterrupt: 

## Cell 14 — Final checkpoint + push to Kaggle

Saves current-step checkpoint (if not already at a 1000-step boundary),
writes MANIFEST + dataset-metadata, pushes to `mhassann/moshi-p3-ckpt`.


In [15]:
import subprocess, json, pathlib, os

OUT_DIR = pathlib.Path("/kaggle/working")
username = os.environ.get("KAGGLE_USERNAME", "mhassann")

final_ckpt = OUT_DIR / f"ckpt_step_{step}.pt"
if not final_ckpt.exists():
    ckpt = {
        "step": step,
        "phase": "P3",
        "wall_seconds": round(time.time() - t_session + total_wall, 1),
        "student_backbone": smol_tt.backbone.state_dict(),
        "in_adapter": smol_tt.in_adapter.state_dict(),
        "out_adapter": smol_tt.out_adapter.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "torch_version": torch.__version__,
    }
    torch.save(ckpt, final_ckpt)
    print(f"Final ckpt: {final_ckpt.name} ({final_ckpt.stat().st_size/1e9:.2f} GB)")
    del ckpt

ckpts = sorted(OUT_DIR.glob("ckpt_step_*.pt"))
if len(ckpts) > 1:
    for old in ckpts[:-1]:
        old.unlink()
        print(f"  deleted old: {old.name}")

manifest = f"""# MANIFEST - moshi-p3-ckpt

Phase 3 attention-polish checkpoint.

| Key | Value |
|---|---|
| step | {step} |
| phase | P3 |
| gate_metric | val_cos_sim |
| gate_target | > 0.85 (already met from Phase 2 exit at 0.900) |
| init_from | mhassann/moshi-p2-ckpt/ckpt_step_611.pt |
| loss | 1.0*hidden_cos + 0.1*text_jsd + 0.05*attn_kl |
| teacher | LIVE (moshiko-pytorch-bf16 backbone + emb only, cuda:1) |
| layer_pairs | 8 paired (student, teacher) layers for attn KL |
"""
(OUT_DIR / "MANIFEST.md").write_text(manifest)

dataset_id = f"{username}/moshi-p3-ckpt"
metadata = {
    "title":    "moshi-p3-ckpt",
    "id":       dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
}
(OUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

print("\nFiles to upload:")
upload_files = ["MANIFEST.md", "train_log.jsonl", "dataset-metadata.json"]
upload_files += [p.name for p in OUT_DIR.glob("ckpt_step_*.pt")]
total_gb = 0.0
for name in upload_files:
    p = OUT_DIR / name
    if p.exists():
        gb = p.stat().st_size / 1e9
        total_gb += gb
        print(f"  {name:<35} {gb*1000:8.1f} MB")
print(f"  {'TOTAL':<35} {total_gb:8.2f} GB")

assert total_gb < 19, f"Upload too large: {total_gb:.1f} GB"

print("\nPushing dataset ...")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(OUT_DIR)],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")
if r.returncode == 0:
    print(f"SUCCESS (create) - kaggle.com/{dataset_id}")
else:
    print(f"Create failed (rc={r.returncode}), trying version bump ...")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUT_DIR),
         "-m", f"P3 step {step}"],
        capture_output=True, text=True,
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
    else:
        print(f"SUCCESS (version) - kaggle.com/{dataset_id}")

print("\n=== Phase-3 session COMPLETE ===")
print(f"Step: {step}, next session resumes from this checkpoint.")


Final ckpt: ckpt_step_1056.pt (6.56 GB)
  deleted old: ckpt_step_1000.pt

Files to upload:
  MANIFEST.md                              0.0 MB
  train_log.jsonl                          0.2 MB
  dataset-metadata.json                    0.0 MB
  ckpt_step_1056.pt                     6561.8 MB
  TOTAL                                   6.56 GB

Pushing dataset ...
Starting upload for file MANIFEST.md
Upload successful: MANIFEST.md (466B)
Starting upload for file ckpt_step_1056.pt
Upload successful: ckpt_step_1056.pt (6GB)
Skipping folder: moshi_repo; use '--dir-mode' to upload folders
Skipping folder: .virtual_documents; use '--dir-mode' to upload folders
Starting upload for file train_log.jsonl
Upload successful: train_log.jsonl (163KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/mhassann/moshi-p3-ckpt

SUCCESS (create) - kaggle.com/mhassann/moshi-p3-ckpt

=== Phase-3 session COMPLETE ===
Step: 1056, next session resumes from this checkpo